In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.document_loaders import PyPDFLoader
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableSequence
from langchain_classic.storage import InMemoryStore
from langchain_classic.retrievers import ParentDocumentRetriever
from dotenv import load_dotenv

/home/orlandojunior/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-13 16:49:18.943726: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-13 16:49:18.978385: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-13 16:49:20.400000: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly differ

In [2]:
load_dotenv()

True

In [3]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model='gpt-3.5-turbo')

In [4]:
pdf_link = "./os-sertoes.pdf"

loader = PyPDFLoader(pdf_link,extract_images=False)
pages = loader.load_and_split()

len(pages)

658

In [5]:
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500
) 
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    add_start_index=True
)

In [6]:
store = InMemoryStore()
vectorstore = Chroma(embedding_function=embeddings,persist_directory="chromadb2")

In [7]:
parent_document_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)

parent_document_retriever.add_documents(pages,id=None)

In [8]:
parent_document_retriever.vectorstore.get()

{'ids': ['62c6f69a-9c46-4844-970a-9704205444db',
  'a9310929-aab3-49b0-9f12-5c07f2143e09',
  '761d3858-9422-43b4-bf66-adeffede937d',
  'cc7bafaa-ed82-4d62-a3e9-8c05ad8481a2',
  '7ce2e4ef-361d-4409-b742-3af002e8f26a',
  '6539ea56-5e48-4033-81a8-870c7ab54186',
  '76ba89ab-7fdd-4c1a-92dd-3917b55af867',
  '1daa0761-001b-47ba-8fcb-e8f3fdee2c87',
  'ea21805a-8881-4d6f-af2b-1e0159f6cfbf',
  '16570852-ce29-48d8-abc4-7986a852c733',
  'a4512e88-2170-4366-b025-fb8c6a2df4e7',
  '1d09fa4f-6103-4733-a495-75fbe26f1e98',
  '90705ddc-4b1c-4214-bd6d-cf55e32621b8',
  '9609874b-93b5-4653-917e-92102e09f562',
  'ff1fa6e2-5182-4f6a-b0d3-b93042df6996',
  '0bacb61b-85f9-4e46-921f-722572faf8ea',
  'e784ae7f-0c02-4556-9979-e5867107d3ae',
  'b72e460f-9d78-43ef-adc4-e972f3709569',
  '9f862aef-8828-4968-8e6c-86da92c06989',
  '0e2efa9a-38a5-4753-9b36-a3d35bc00f36',
  'dcf0d92d-a68d-4195-8755-94f1780357f3',
  'e5c4d2ce-d40a-4683-ad38-2d7e8ba85bbe',
  'cdcc0bc8-73d2-42de-9e5c-fd3cc8f445b2',
  '1909b29f-6e86-4866-a877-

In [9]:
TEMPLATE = """
    Você é um especialista no livro "os sertões. Responda a pergunta utilizando o contexto informado"
    Query:
    {question}

    Context:
    {context}
"""
prompt = PromptTemplate(input_variables=["context", "question"], template=TEMPLATE)

In [10]:
sequence = RunnableSequence(prompt | llm | StrOutputParser())

In [11]:

questions = [
    "Qual é a visão de Euclides da Cunha sobre o ambiente natural do sertão nordestino e como ele influencia a vida dos habitantes?",
    "Quais são as principais características da população sertaneja descritas por Euclides da Cunha? Como ele relaciona essas características com o ambiente em que vivem?",
    "Qual foi o contexto histórico e político que levou à Guerra de Canudos, segundo Euclides da Cunha?",
    "Como Euclides da Cunha descreve a figura de Antônio Conselheiro e seu papel na Guerra de Canudos?",
    "Quais são os principais aspectos da crítica social e política presentes em 'Os Sertões'? Como esses aspectos refletem a visão do autor sobre o Brasil da época?"
]

for i,question in enumerate(questions):
    print(f'Questão {i} : {question}')
    print("Resposta:")
    print("\n")
    context = parent_document_retriever.invoke(question)
    response = sequence.invoke({"context":context, "question": question})
    print(response)
    print("=============================")


Questão 0 : Qual é a visão de Euclides da Cunha sobre o ambiente natural do sertão nordestino e como ele influencia a vida dos habitantes?
Resposta:


Euclides da Cunha apresenta em "Os Sertões" uma visão do ambiente natural do sertão nordestino como um lugar extremamente hostil e desafiador. Ele descreve os caracteres geológicos, topográficos e as influências físicas do local de forma a destacar a dureza do clima e da vegetação, como a caatinga. Essas condições adversas influenciam diretamente a vida dos habitantes do sertão, tornando-a difícil e muitas vezes trágica. Euclides da Cunha mostra como as condições climáticas e naturais do sertão nordestino moldam a cultura e a sociedade da região, interferindo profundamente na vida cotidiana e nas relações entre as pessoas que habitam esse ambiente inóspito.
Questão 1 : Quais são as principais características da população sertaneja descritas por Euclides da Cunha? Como ele relaciona essas características com o ambiente em que vivem?
Respo